# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/viki22uied/ML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring**

I'm drawn to this lane because I want to build systems that reduce real workload, not just produce a score for its own sake and this lane is literally that a ranked queue that tells a reviewer where to spend their limited time first. It also builds directly on Week 1 to 2 work I already validated: I found that raw search volume barely predicts traffic (correlation ≈ 0.00) and that a simple hand rule looked strong in-sample but its advantage shrank on holdout data. That gap between "looks good" and "actually generalizes" is exactly the kind of problem worth solving with a properly validated ranking system, which is what this lane produces.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess

REPO_URL = "https://github.com/viki22uied/ML"
REPO_DIR = "ML"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())
print(os.listdir())


Now in: /content/ML/ML
['data', 'GUIDE.md', 'notebooks', 'submission', 'skills', 'DATA_USE.md', '.github', '.git', 'scripts', 'SETUP.md', 'requirements.txt', 'CLAUDE.md', 'outputs', 'LICENSE', 'README.md', 'docs', 'AGENTS.md', '.gitignore', 'work']


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages should a content reviewer look at first this week, out of thousands, given limited review capacity (e.g. can only review ~50 pages/week)?

**Unit of analysis:** one page (content_id), scored and ranked at a point in time.

**Action:** a reviewer opens the top-ranked pages and decides — refresh the content, protect it as-is, or deprioritize it — instead of reviewing pages in an arbitrary or purely recency-based order.

**Cost of a wrong call:**
- False positive (flagged as needing review, but it's actually fine): wastes a reviewer's limited time — the cost is reviewer-hours, not revenue.
- False negative (a genuinely declining, high-traffic page never surfaces in the queue): the page keeps losing visibility/traffic silently until someone notices by accident — a slower, quieter, but potentially larger cost, especially for high-`impressions_90d` pages.
- Given limited review capacity, Precision@K (K = weekly review capacity) matches the real decision better than overall accuracy — a model can look accurate overall while still picking a bad top-50.

**Why data/ML helps, not just a rule:** my Week 2 hand rule (stale × visible) got strong in-sample Precision@20 (0.900), but that's a fixed formula that can't adapt to interactions a human wouldn't think to hand-code — the tree's actual first split was `impressions_90d`, not staleness. The gap between in-sample and holdout performance (0.900 in-sample vs. 0.400 holdout @20) is itself worth investigating further — a rule that "sounds right" can still overstate itself, and only careful validation catches that.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Loaded the starter dataset (30,000 pages). 54.2% of pages are currently labeled "declining" — a near-even split, not a rare event. Pages that are both stale (not updated in 180+ days) and still visible (500+ impressions in 90 days) are surprisingly rare — just 17 pages (0.1%) — so that specific combination alone won't be a large enough queue on its own; other reason codes will matter more for volume. More interesting: declining pages have a noticeably higher median impression count (961) than non-declining pages (472) — meaning decline isn't concentrated in low-traffic, low-stakes pages. The pages losing ground tend to be the ones with more visibility to lose, which is exactly why a ranked review queue matters more than a blanket rule.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n_pages = df.shape[0]
decline_rate = (df["trend_direction"].str.lower() == "down").mean()
print(f"Pages: {n_pages} | declining rate: {decline_rate:.3f}")

stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"Stale AND visible pages: {stale_visible} ({stale_visible/n_pages:.1%} of all pages)")

declining = df[df["trend_direction"].str.lower() == "down"]
not_declining = df[df["trend_direction"].str.lower() != "down"]
print(f"Median impressions_90d, declining pages: {declining['impressions_90d'].median():.0f}")
print(f"Median impressions_90d, non-declining pages: {not_declining['impressions_90d'].median():.0f}")

Pages: 30000 | declining rate: 0.542
Stale AND visible pages: 17 (0.1% of all pages)
Median impressions_90d, declining pages: 961
Median impressions_90d, non-declining pages: 472


## 4. Careful words: what I can and can't claim

**What this work can say:**
- Observed associations between page characteristics (staleness, visibility, position, engagement) and a current-window decline label.
- Directional evidence about which signals rank pages usefully for review — validated on client holdout, not just in-sample.
- Decision-support: a ranked queue that helps a reviewer spend limited time on the most promising candidates first.

**What this work cannot say:**
- That refreshing a page *causes* recovery — that requires an actual experiment (before/after with a control group), which this dataset doesn't provide.
- That any signal here reflects a real Google ranking factor — these are FlyRank's own observable search/analytics signals, not Google's algorithm.
- That the current `is_declining_label` is the ideal target — it's a current-window proxy, not a genuine future outcome. A stronger version of this lane (possible by Week 4) would predict a future window's decline from a prior window's features.
- That precision numbers from the 30k-row starter slice generalize to the full ~79M-row warehouse without re-validating there.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.